# 03. EDA y estadisticas

Esta notebook desarrolla una capa de exploracion descriptiva y estadistica usando la version reducida del dataset. El objetivo es entender estructura, perfiles principales y relaciones basicas antes de pasar a la notebook de visualizaciones.

## Alcance

- cargar la version reducida del dataset limpio
- revisar estructura general y calidad final
- resumir variables numericas clave
- perfilar categorias principales
- explorar tecnologias mas frecuentes desde columnas multiseleccion
- comparar salario y experiencia por grupos relevantes
- obtener una vista simple de correlaciones numericas
- exportar tablas de resumen para uso posterior

**Nota:** esta notebook privilegia tablas y estadisticos. Las visualizaciones quedan separadas en la siguiente etapa del portafolio.


## Subpaso 1. Carga del dataset reducido

**Proposito:** trabajar sobre una version enfocada del dataset, ya depurada y alineada con el analisis posterior.

**Resultado esperado:** acceso al archivo `data/cleaned_outputs/survey_data_cleaned_reduced.csv` y a una carpeta `data/eda_outputs` para exportar tablas.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_PATH = DATA_DIR / "cleaned_outputs" / "survey_data_cleaned_reduced.csv"
OUTPUT_DIR = DATA_DIR / "eda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)
INPUT_PATH, OUTPUT_DIR


## Subpaso 2. Vista estructural del dataset

**Proposito:** confirmar que la version reducida tenga el alcance prometido por la notebook de limpieza.

**Resultado:** el dataset reducido contiene `18,845` filas y `32` columnas, con enfoque en demografia, experiencia, salario, satisfaccion y stacks tecnologicos.

**Hallazgo:** ya no quedan faltantes, por lo que el EDA puede concentrarse en distribuciones y relaciones, no en calidad basica de datos.


In [ ]:
structure_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "non_null_count": df.count(),
        "missing_values": df.isna().sum(),
    }
)

print("Shape:", df.shape)
print("Total missing values:", int(df.isna().sum().sum()))
structure_summary


## Subpaso 3. Resumen numerico de salario, experiencia y satisfaccion

**Proposito:** revisar tendencia central, dispersion y rangos de las variables numericas mas importantes.

**Resultado:** `ConvertedCompYearly`, `WorkExp`, `JobSat` y `JobSatPoints_6` resumen la parte cuantitativa principal del proyecto.

**Hallazgos:** la compensacion anual sigue mostrando alta dispersion; la experiencia laboral tiene una mediana de `10` anos; `JobSat` se concentra en valores altos. `CompTotal` conserva valores extremos y debe leerse con cautela porque mezcla monedas y escalas locales.


In [ ]:
numeric_columns = [
    "CompTotal",
    "ConvertedCompYearly",
    "WorkExp",
    "JobSat",
    "JobSatPoints_6",
    "ConvertedCompYearly_MinMax",
    "ConvertedCompYearly_Zscore",
]

numeric_summary = df[numeric_columns].describe().transpose()
numeric_summary


## Subpaso 4. Perfil de categorias principales

**Proposito:** identificar quienes componen la muestra en terminos de edad, modalidad de trabajo, empleo, pais, tipo de desarrollador e industria.

**Resultado:** se generan tablas de frecuencia para los segmentos principales del dataset.

**Hallazgos:** el grupo dominante es `25-34 years old`; el empleo full-time predomina claramente; `Remote` y `Hybrid` concentran la mayor parte de respuestas; `United States`, `Germany`, `India` y `United Kingdom` lideran por pais; `Developer, full-stack` domina la categoria de rol; `Software Development` es la industria mas representada.


In [ ]:
age_distribution = df["Age"].value_counts().rename_axis("Age").reset_index(name="count")
employment_distribution = df["Employment"].value_counts().rename_axis("Employment").reset_index(name="count")
remotework_distribution = df["RemoteWork"].value_counts().rename_axis("RemoteWork").reset_index(name="count")
country_top10 = df["Country"].value_counts().head(10).rename_axis("Country").reset_index(name="count")
devtype_top10 = df["DevType"].value_counts().head(10).rename_axis("DevType").reset_index(name="count")
industry_top10 = df["Industry"].value_counts().head(10).rename_axis("Industry").reset_index(name="count")

print("Age distribution")
display(age_distribution.head(8))
print("Remote work distribution")
display(remotework_distribution)
print("Top countries")
display(country_top10)


## Subpaso 5. Tecnologias mas frecuentes en el survey

**Proposito:** transformar columnas multiseleccion en conteos interpretables para lenguaje, bases de datos, plataformas y frameworks.

**Resultado:** se generan tablas Top 10 para uso actual y deseo futuro.

**Hallazgos:** `JavaScript`, `SQL`, `HTML/CSS`, `TypeScript` y `Python` lideran entre lenguajes; `PostgreSQL` domina tanto el uso actual como la preferencia futura; `AWS`, `Azure` y `Google Cloud` lideran en plataformas; `React` y `Node.js` dominan en frameworks y mantienen fuerza en preferencia futura.


In [ ]:
def top_multiselect_counts(series: pd.Series, top_n: int = 10, label: str = "item") -> pd.DataFrame:
    cleaned = series.dropna().astype(str)
    cleaned = cleaned[~cleaned.isin(["", "Not specified"])]
    exploded = cleaned.str.split(";").explode().str.strip()
    counts = exploded.value_counts().head(top_n)
    return counts.rename_axis(label).reset_index(name="count")

current_languages_top10 = top_multiselect_counts(df["LanguageHaveWorkedWith"], label="language")
future_languages_top10 = top_multiselect_counts(df["LanguageWantToWorkWith"], label="language")
current_databases_top10 = top_multiselect_counts(df["DatabaseHaveWorkedWith"], label="database")
future_databases_top10 = top_multiselect_counts(df["DatabaseWantToWorkWith"], label="database")
current_platforms_top10 = top_multiselect_counts(df["PlatformHaveWorkedWith"], label="platform")
future_platforms_top10 = top_multiselect_counts(df["PlatformWantToWorkWith"], label="platform")
current_webframes_top10 = top_multiselect_counts(df["WebframeHaveWorkedWith"], label="web_framework")
future_webframes_top10 = top_multiselect_counts(df["WebframeWantToWorkWith"], label="web_framework")

print("Current languages")
display(current_languages_top10)
print("Future databases")
display(future_databases_top10)


## Subpaso 6. Comparaciones por grupos

**Proposito:** contrastar salario y experiencia entre segmentos relevantes sin entrar todavia en graficos.

**Resultado:** se generan tablas comparativas por `RemoteWork` y `Age`.

**Hallazgos:** la modalidad `Remote` muestra el promedio de compensacion anual mas alto del grupo; la media salarial tiende a crecer con la edad hasta los tramos senior. La mediana aparece muy parecida entre grupos porque `ConvertedCompYearly` fue imputada con una mediana global en la etapa de limpieza, asi que las medias y los tamanos de grupo son mas informativos aqui.


In [ ]:
salary_by_remotework = (
    df.groupby("RemoteWork")["ConvertedCompYearly"]
    .agg(["count", "median", "mean"])
    .sort_values("mean", ascending=False)
    .reset_index()
)

salary_by_age = (
    df.groupby("Age")["ConvertedCompYearly"]
    .agg(["count", "median", "mean"])
    .sort_values("mean", ascending=False)
    .reset_index()
)

workexp_by_age = (
    df.groupby("Age")["WorkExp"]
    .agg(["count", "median", "mean"])
    .sort_values("mean", ascending=False)
    .reset_index()
)

display(salary_by_remotework)
display(salary_by_age)


## Subpaso 7. Correlacion numerica basica

**Proposito:** obtener una lectura rapida de relacion lineal entre salario, experiencia y satisfaccion.

**Resultado:** se calcula una matriz de correlacion entre variables numericas clave.

**Hallazgos:** `WorkExp` muestra una relacion positiva pero moderada con `ConvertedCompYearly`; `JobSat` y `JobSatPoints_6` presentan relaciones debiles con salario y experiencia. Esto sugiere que, en este recorte, la experiencia explica parte del salario, mientras que la satisfaccion no muestra una asociacion lineal fuerte.


In [ ]:
correlation_columns = [
    "CompTotal",
    "ConvertedCompYearly",
    "WorkExp",
    "JobSat",
    "JobSatPoints_6",
    "ConvertedCompYearly_MinMax",
    "ConvertedCompYearly_Zscore",
]

correlation_matrix = df[correlation_columns].corr(numeric_only=True)
correlation_matrix


## Subpaso 8. Exportacion de tablas de EDA

**Proposito:** dejar salidas reutilizables para visualizaciones, dashboarding y documentacion del caso.

**Resultado:** se exportan tablas descriptivas, perfiles categoricos, conteos de tecnologias, comparaciones por grupos y matriz de correlacion.

**Hallazgo operativo:** separar estas salidas reduce retrabajo y hace mas transparente que resultados vienen de EDA tabular y cuales pertenecen a la etapa de visualizacion.


In [ ]:
structure_summary.reset_index(names="column_name").to_csv(OUTPUT_DIR / "eda_structure_summary.csv", index=False)
numeric_summary.to_csv(OUTPUT_DIR / "eda_numeric_summary.csv")
age_distribution.to_csv(OUTPUT_DIR / "eda_age_distribution.csv", index=False)
employment_distribution.to_csv(OUTPUT_DIR / "eda_employment_distribution.csv", index=False)
remotework_distribution.to_csv(OUTPUT_DIR / "eda_remotework_distribution.csv", index=False)
country_top10.to_csv(OUTPUT_DIR / "eda_country_top10.csv", index=False)
devtype_top10.to_csv(OUTPUT_DIR / "eda_devtype_top10.csv", index=False)
industry_top10.to_csv(OUTPUT_DIR / "eda_industry_top10.csv", index=False)
current_languages_top10.to_csv(OUTPUT_DIR / "eda_current_languages_top10.csv", index=False)
future_languages_top10.to_csv(OUTPUT_DIR / "eda_future_languages_top10.csv", index=False)
current_databases_top10.to_csv(OUTPUT_DIR / "eda_current_databases_top10.csv", index=False)
future_databases_top10.to_csv(OUTPUT_DIR / "eda_future_databases_top10.csv", index=False)
current_platforms_top10.to_csv(OUTPUT_DIR / "eda_current_platforms_top10.csv", index=False)
future_platforms_top10.to_csv(OUTPUT_DIR / "eda_future_platforms_top10.csv", index=False)
current_webframes_top10.to_csv(OUTPUT_DIR / "eda_current_webframes_top10.csv", index=False)
future_webframes_top10.to_csv(OUTPUT_DIR / "eda_future_webframes_top10.csv", index=False)
salary_by_remotework.to_csv(OUTPUT_DIR / "eda_salary_by_remotework.csv", index=False)
salary_by_age.to_csv(OUTPUT_DIR / "eda_salary_by_age.csv", index=False)
workexp_by_age.to_csv(OUTPUT_DIR / "eda_workexp_by_age.csv", index=False)
correlation_matrix.to_csv(OUTPUT_DIR / "eda_correlation_matrix.csv")

print("Saved EDA tables to:")
print(OUTPUT_DIR)


## Cierre de la etapa

Esta notebook deja un mapa descriptivo del dataset reducido: quien responde, que tecnologias dominan, como se distribuyen salario y experiencia, y que relaciones numericas aparecen de forma preliminar. La siguiente notebook se enfocara en traducir estos hallazgos a visualizaciones claras para el portafolio.
